# M04-P02 — Multimodal Incident Analyzer

En M01-P02 construimos `Analyze` para convertir **texto libre** en una salida estructurada y validada.

Ahora vamos a ampliar la misma capacidad para trabajar con:

```text
texto + imagen
```

Caso de uso:

```text
descripción de incidencia
+
captura de dashboard / error / activo
        ↓
modelo multimodal
        ↓
análisis estructurado
```

La práctica utiliza un modelo multimodal real de Amazon Bedrock mediante la API Converse.

## 1. Qué queremos demostrar

No queremos hacer una demo aislada de reconocimiento de imágenes.

Queremos comprobar qué aporta una evidencia visual a una aplicación empresarial que ya existe.

Durante la práctica:

1. generaremos una captura sintética de un dashboard;
2. enviaremos la imagen a Bedrock para comprobar qué puede observar;
3. compararemos **texto solo** frente a **texto + imagen**;
4. separaremos hechos textuales, evidencia visual e hipótesis;
5. integraremos la nueva capacidad en `Enterprise GenAI Assistant`.

In [ ]:
from pathlib import Path
import json
import time

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from enterprise_genai.config import get_settings
from enterprise_genai.analysis import (
    analyze_incident,
    analyze_incident_multimodal,
)

settings = get_settings()
session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=settings.aws_region,
)
runtime = session.client("bedrock-runtime")
identity = session.client("sts").get_caller_identity()
MODEL_ID = settings.bedrock_text_model_id

print("AWS identity:", identity["Arn"])
print("Region:", settings.aws_region)
print("Model:", MODEL_ID)

## 2. Crear una evidencia visual controlada

Para que todos trabajemos con la misma evidencia vamos a generar un dashboard sintético.

La captura añadirá información que **no aparece explícitamente** en el texto de la incidencia:

- crecimiento de errores de autenticación a partir de las 08:15;
- estado `DEGRADED` del backend de autenticación;
- servicio web en estado `OK`.

Esto nos permitirá comprobar si el modelo utiliza realmente la imagen.

In [ ]:
OUT_DIR = Path("outputs/m04_p02")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARD_PATH = OUT_DIR / "vpn_auth_dashboard.png"

times = ["07:50","08:00","08:05","08:10","08:15","08:20","08:25","08:30"]
auth_errors = [1, 1, 2, 2, 12, 31, 46, 48]

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 3, height_ratios=[2.2, 1])

ax = fig.add_subplot(gs[0, :])
ax.plot(times, auth_errors, marker="o", linewidth=2)
ax.axvline("08:15", linestyle="--", alpha=0.7)
ax.set_title("ACME — Remote Access Operations Dashboard")
ax.set_ylabel("VPN authentication failures / min")
ax.set_xlabel("Time")
ax.grid(alpha=0.25)

cards = [
    ("VPN AUTH BACKEND", "DEGRADED", "Intermittent authentication errors"),
    ("CORPORATE WEB", "OK", "No active alerts"),
    ("VPN GATEWAY", "OK", "Traffic within expected range"),
]

for i, (title, status, detail) in enumerate(cards):
    a = fig.add_subplot(gs[1, i])
    a.axis("off")
    a.text(0.5, 0.78, title, ha="center", va="center", fontsize=12, weight="bold")
    a.text(0.5, 0.46, status, ha="center", va="center", fontsize=18, weight="bold")
    a.text(0.5, 0.18, detail, ha="center", va="center", fontsize=9)

plt.tight_layout()
fig.savefig(DASHBOARD_PATH, dpi=150, bbox_inches="tight")
plt.show()

print("Imagen:", DASHBOARD_PATH)
print("Tamaño:", DASHBOARD_PATH.stat().st_size, "bytes")

## 3. Abrir primero la caja: una petición multimodal sin estructura

Antes de pedir JSON estructurado vamos a comprobar que el modelo puede inspeccionar la imagen.

La petición contiene dos bloques:

```text
content
├── text
└── image
```

Con `boto3`, los bytes de la imagen se envían directamente; no necesitamos convertirlos manualmente a Base64.

In [ ]:
image_bytes = DASHBOARD_PATH.read_bytes()

started = time.perf_counter()
raw_response = runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Analiza únicamente lo que sea directamente visible en esta captura. "
                        "Enumera evidencias observables y evita formular una causa raíz. "
                        "Si una cifra no puede leerse con claridad, no la inventes."
                    )
                },
                {
                    "image": {
                        "format": "png",
                        "source": {"bytes": image_bytes},
                    }
                },
            ],
        }
    ],
    inferenceConfig={"temperature": 0.0, "maxTokens": 350},
)
client_ms = (time.perf_counter() - started) * 1000

raw_text = raw_response["output"]["message"]["content"][0]["text"]
print(raw_text)
print()
print("Usage:", raw_response.get("usage", {}))
print("Bedrock metrics:", raw_response.get("metrics", {}))
print(f"Client time: {client_ms:.0f} ms")

### Qué debemos observar

El modelo debería poder extraer evidencias como:

- aumento de fallos de autenticación;
- momento aproximado en el que comienza el crecimiento;
- backend de autenticación degradado;
- web corporativa sin alertas;
- gateway VPN en estado normal.

Pero no debería convertir automáticamente esas observaciones en una causa raíz confirmada.

## 4. Texto solo: nuestro punto de partida de M01-P02

Usaremos una incidencia deliberadamente incompleta:

> Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN.  
> El servicio web corporativo funciona correctamente.  
> El equipo de operaciones está investigando.

El texto **no dice** que el backend de autenticación esté degradado.

Primero ejecutamos el analizador original.

In [ ]:
INCIDENT = (
    "Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. "
    "El servicio web corporativo funciona correctamente. "
    "El equipo de operaciones está investigando."
)

text_only = analyze_incident(INCIDENT)
text_analysis = text_only["analysis"]

print(json.dumps(text_analysis.model_dump(), indent=2, ensure_ascii=False))
print()
print("Stop reason:", text_only["stop_reason"])
print("Usage:", text_only["usage"])

## 5. Texto + imagen: análisis multimodal estructurado

Ahora enviamos exactamente el mismo texto junto con la captura.

Ampliaremos el schema con campos específicos para visión:

```text
visual_summary
visual_evidence
visual_uncertainties
text_image_consistency
```

La regla es importante:

```text
observed_facts
→ hechos que podemos sostener con el conjunto de evidencias

visual_evidence
→ información directamente observable en la imagen

hypotheses
→ posibles explicaciones, no hechos
```

In [ ]:
multi = analyze_incident_multimodal(
    INCIDENT,
    image_bytes=image_bytes,
    image_format="png",
)
multi_analysis = multi["analysis"]

print(json.dumps(multi_analysis.model_dump(), indent=2, ensure_ascii=False))
print()
print("Stop reason:", multi["stop_reason"])
print("Usage:", multi["usage"])
print("Bedrock metrics:", multi["metrics"])

## 6. Comparar qué aporta la imagen

El objetivo no es que la respuesta multimodal sea simplemente “más larga”.

La pregunta es:

> ¿Aparecen hechos útiles que no podían obtenerse del texto?

Construimos una comparación compacta.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "enfoque": "Texto",
            "categoria": text_analysis.category,
            "servicio": text_analysis.affected_service,
            "usuarios": text_analysis.affected_users,
            "n_hechos": len(text_analysis.observed_facts),
            "evidencia_visual": "No aplica",
            "consistencia": "No aplica",
            "tokens": text_only["usage"].get("totalTokens"),
        },
        {
            "enfoque": "Texto + imagen",
            "categoria": multi_analysis.category,
            "servicio": multi_analysis.affected_service,
            "usuarios": multi_analysis.affected_users,
            "n_hechos": len(multi_analysis.observed_facts),
            "evidencia_visual": len(multi_analysis.visual_evidence),
            "consistencia": multi_analysis.text_image_consistency,
            "tokens": multi["usage"].get("totalTokens"),
        },
    ]
)

display(comparison)

## 7. Inspeccionar solo la evidencia visual

Queremos que sea visible qué información proviene de la captura y qué parte es razonamiento del modelo.

In [ ]:
print("Resumen visual:")
print(multi_analysis.visual_summary)

print("\nEvidencias visuales:")
for item in multi_analysis.visual_evidence:
    print("-", item)

print("\nIncertidumbres visuales:")
if multi_analysis.visual_uncertainties:
    for item in multi_analysis.visual_uncertainties:
        print("-", item)
else:
    print("- Ninguna declarada")

print("\nConsistencia texto/imagen:", multi_analysis.text_image_consistency)

## 8. Hecho visual no significa causa raíz

Supongamos que la captura muestra:

```text
VPN AUTH BACKEND = DEGRADED
```

Eso permite afirmar:

```text
el dashboard muestra el backend degradado
```

Pero no permite afirmar automáticamente:

```text
la causa raíz es X
```

El estado visual es una evidencia.

La causa raíz sigue necesitando investigación, logs, métricas y otras fuentes.

## 9. Reto opcional — contradicción entre modalidades

Modifica el dashboard para que `CORPORATE WEB` aparezca como `DEGRADED` mientras el texto sigue diciendo que funciona correctamente.

Después vuelve a ejecutar el análisis multimodal.

¿Qué esperas que ocurra con:

```text
text_image_consistency
```

?

La aplicación no debería ocultar una contradicción entre dos evidencias; debería hacerla visible y solicitar revisión humana.

## 10. Integración en Enterprise GenAI Assistant

La página `Analyze` ya acepta una imagen opcional.

Sin imagen:

```text
texto
→ Structured Incident Analyzer
```

Con imagen:

```text
texto + imagen
→ Multimodal Incident Analyzer
```

La aplicación mostrará además:

- evidencia visual;
- resumen visual;
- consistencia texto/imagen;
- incertidumbres visuales;
- métricas y JSON estructurado.

Arranca la aplicación:

```bash
streamlit run app/Home.py
```

y abre **Analyze**.

## 11. Qué debes llevarte

La evolución de la aplicación es:

```text
M01-P02
texto
→ análisis estructurado

M04-P02
texto + imagen
→ análisis multimodal estructurado
```

La imagen no es un adorno.

Puede convertirse en una evidencia adicional que:

- confirme el texto;
- aporte información nueva;
- contradiga otras fuentes;
- aumente la necesidad de revisión humana.

Ésta es la utilidad real de incorporar visión a una aplicación empresarial.